# Task 2 — Model Building and Training

This notebook:
- Loads the preprocessed e‑commerce and credit card data
- Trains Logistic Regression (baseline), Random Forest, and XGBoost
- Performs hyper‑parameter tuning with GridSearchCV
- Validates using Stratified K‑Fold (k=5) and reports mean ± std of AUC‑PR and F1
- Compares models side‑by‑side and selects the best for each dataset

In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    average_precision_score,
    precision_recall_curve,
    auc,
    roc_auc_score
)

In [2]:
# E‑commerce data
with open('../data/processed/fraud_train.pkl', 'rb') as f:
    X_train_f, y_train_f = pickle.load(f)
with open('../data/processed/fraud_test.pkl', 'rb') as f:
    X_test_f, y_test_f, preprocessor_f = pickle.load(f)

# Credit card data
with open('../data/processed/creditcard_train.pkl', 'rb') as f:
    X_train_c, y_train_c = pickle.load(f)
with open('../data/processed/creditcard_test.pkl', 'rb') as f:
    X_test_c, y_test_c = pickle.load(f)

print('E‑commerce shapes:', X_train_f.shape, X_test_f.shape)
print('Credit card shapes:', X_train_c.shape, X_test_c.shape)

E‑commerce shapes: (219136, 192) (30223, 192)
Credit card shapes: (453204, 30) (56746, 30)


In [3]:
def evaluate_model(name, model, X_test, y_test):
    """Print F1, AUC‑PR, and confusion matrix for a trained model."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
    f1 = f1_score(y_test, y_pred)
    ap = average_precision_score(y_test, y_prob)
    print(f"{name} | F1: {f1:.4f} | AUC‑PR: {ap:.4f}")
    print(confusion_matrix(y_test, y_pred))
    return f1, ap

def cv_evaluate(name, model, X, y, cv=5):
    """Stratified K‑Fold cross‑validation with F1 and AUC‑PR."""
    scoring = {'f1': 'f1', 'avg_precision': 'average_precision'}
    cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    f1_mean = np.mean(cv_results['test_f1'])
    f1_std = np.std(cv_results['test_f1'])
    ap_mean = np.mean(cv_results['test_avg_precision'])
    ap_std = np.std(cv_results['test_avg_precision'])
    print(f"{name} CV | F1: {f1_mean:.4f} ± {f1_std:.4f} | AUC‑PR: {ap_mean:.4f} ± {ap_std:.4f}")
    return f1_mean, f1_std, ap_mean, ap_std

In [4]:
# Logistic Regression baseline (e‑commerce)
lr_f = LogisticRegression(max_iter=1000, random_state=42)
lr_f.fit(X_train_f, y_train_f)
evaluate_model('Logistic Regression (E‑comm)', lr_f, X_test_f, y_test_f)
cv_evaluate('Logistic Regression (E‑comm)', lr_f, X_train_f, y_train_f)

Logistic Regression (E‑comm) | F1: 0.2776 | AUC‑PR: 0.3911
[[17927  9466]
 [  848  1982]]
Logistic Regression (E‑comm) CV | F1: 0.6891 ± 0.0036 | AUC‑PR: 0.8009 ± 0.0030


(np.float64(0.68908030392779),
 np.float64(0.0036150610342170356),
 np.float64(0.8009286679600806),
 np.float64(0.003000941879922183))

In [5]:
from sklearn.model_selection import RandomizedSearchCV

# Reduced parameter grid
rf_f_dist = {
    'n_estimators': [100, 150],
    'max_depth': [10, None]
}
rf_f_base = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_f_search = RandomizedSearchCV(
    rf_f_base,
    param_distributions=rf_f_dist,
    n_iter=3,                 # try only 3 random combinations
    scoring='f1',
    cv=2,                     # 2‑fold CV is enough for this exercise
    n_jobs=2,                 # use only 2 cores so your machine stays responsive
    random_state=42
)
rf_f_search.fit(X_train_f, y_train_f)
best_rf_f = rf_f_search.best_estimator_
print('Best RF params:', rf_f_search.best_params_)
evaluate_model('Random Forest (E‑comm)', best_rf_f, X_test_f, y_test_f)
cv_evaluate('Random Forest (E‑comm)', best_rf_f, X_train_f, y_train_f)

Best RF params: {'n_estimators': 150, 'max_depth': None}
Random Forest (E‑comm) | F1: 0.6941 | AUC‑PR: 0.6293
[[27346    47]
 [ 1301  1529]]
Random Forest (E‑comm) CV | F1: 0.9708 ± 0.0484 | AUC‑PR: 0.9966 ± 0.0065


(np.float64(0.9707615958465476),
 np.float64(0.048351008842161626),
 np.float64(0.9966096192004569),
 np.float64(0.0065387773369738654))

In [6]:
xgb_f_dist = {
    'n_estimators': [100, 150],
    'learning_rate': [0.1, 0.2],
    'max_depth': [6]
}
scale_pos_weight_f = (len(y_train_f) - sum(y_train_f)) / sum(y_train_f)
xgb_f_base = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight_f,
                           use_label_encoder=False, eval_metric='logloss')
xgb_f_search = RandomizedSearchCV(
    xgb_f_base,
    param_distributions=xgb_f_dist,
    n_iter=3,
    scoring='f1',
    cv=2,
    n_jobs=2,
    random_state=42
)
xgb_f_search.fit(X_train_f, y_train_f)
best_xgb_f = xgb_f_search.best_estimator_
print('Best XGB params:', xgb_f_search.best_params_)
evaluate_model('XGBoost (E‑comm)', best_xgb_f, X_test_f, y_test_f)
cv_evaluate('XGBoost (E‑comm)', best_xgb_f, X_train_f, y_train_f)

Best XGB params: {'n_estimators': 150, 'max_depth': 6, 'learning_rate': 0.2}
XGBoost (E‑comm) | F1: 0.6946 | AUC‑PR: 0.6229
[[27344    49]
 [ 1298  1532]]
XGBoost (E‑comm) CV | F1: 0.9595 ± 0.0531 | AUC‑PR: 0.9829 ± 0.0292


(np.float64(0.9595112985939306),
 np.float64(0.05310943990998698),
 np.float64(0.9829349290443178),
 np.float64(0.029164401455306684))

In [7]:
lr_c = LogisticRegression(max_iter=1000, random_state=42)
lr_c.fit(X_train_c, y_train_c)
evaluate_model('Logistic Regression (Credit)', lr_c, X_test_c, y_test_c)
cv_evaluate('Logistic Regression (Credit)', lr_c, X_train_c, y_train_c)

Logistic Regression (Credit) | F1: 0.1002 | AUC‑PR: 0.6768
[[55172  1479]
 [   12    83]]
Logistic Regression (Credit) CV | F1: 0.9464 ± 0.0006 | AUC‑PR: 0.9920 ± 0.0001


(np.float64(0.9463845558899898),
 np.float64(0.0005703967631745903),
 np.float64(0.9919766785651613),
 np.float64(7.958631213754176e-05))

In [8]:
rf_c_dist = {'n_estimators': [100, 150], 'max_depth': [10, None]}
rf_c_base = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_c_search = RandomizedSearchCV(rf_c_base, param_distributions=rf_c_dist,
                                 n_iter=3, scoring='f1', cv=2, n_jobs=2, random_state=42)
rf_c_search.fit(X_train_c, y_train_c)
best_rf_c = rf_c_search.best_estimator_
print('Best RF Credit params:', rf_c_search.best_params_)
evaluate_model('Random Forest (Credit)', best_rf_c, X_test_c, y_test_c)
cv_evaluate('Random Forest (Credit)', best_rf_c, X_train_c, y_train_c)

Best RF Credit params: {'n_estimators': 150, 'max_depth': None}
Random Forest (Credit) | F1: 0.8229 | AUC‑PR: 0.8133
[[56643     8]
 [   23    72]]
Random Forest (Credit) CV | F1: 0.9999 ± 0.0000 | AUC‑PR: 1.0000 ± 0.0000


(np.float64(0.9999029244508696),
 np.float64(3.651598198751344e-05),
 np.float64(0.999998373573303),
 np.float64(4.4264777103621185e-07))

In [9]:
scale_pos_weight_c = (len(y_train_c) - sum(y_train_c)) / sum(y_train_c)
xgb_c_dist = {'n_estimators': [100, 150], 'learning_rate': [0.1, 0.2], 'max_depth': [6]}
xgb_c_base = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight_c,
                           use_label_encoder=False, eval_metric='logloss')
xgb_c_search = RandomizedSearchCV(xgb_c_base, param_distributions=xgb_c_dist,
                                  n_iter=3, scoring='f1', cv=2, n_jobs=2, random_state=42)
xgb_c_search.fit(X_train_c, y_train_c)
best_xgb_c = xgb_c_search.best_estimator_
print('Best XGB Credit params:', xgb_c_search.best_params_)
evaluate_model('XGBoost (Credit)', best_xgb_c, X_test_c, y_test_c)
cv_evaluate('XGBoost (Credit)', best_xgb_c, X_train_c, y_train_c)

Best XGB Credit params: {'n_estimators': 150, 'max_depth': 6, 'learning_rate': 0.2}
XGBoost (Credit) | F1: 0.7600 | AUC‑PR: 0.8105
[[56622    29]
 [   19    76]]
XGBoost (Credit) CV | F1: 0.9998 ± 0.0000 | AUC‑PR: 1.0000 ± 0.0000


(np.float64(0.999768370941496),
 np.float64(2.8754456472819727e-05),
 np.float64(0.9999880011163397),
 np.float64(5.208772068546654e-06))

In [10]:
results = []
for name, model, X_test, y_test in [
    ('Logistic Regression (E‑comm)', lr_f, X_test_f, y_test_f),
    ('Random Forest (E‑comm)', best_rf_f, X_test_f, y_test_f),
    ('XGBoost (E‑comm)', best_xgb_f, X_test_f, y_test_f),
    ('Logistic Regression (Credit)', lr_c, X_test_c, y_test_c),
    ('Random Forest (Credit)', best_rf_c, X_test_c, y_test_c),
    ('XGBoost (Credit)', best_xgb_c, X_test_c, y_test_c)
]:
    y_prob = model.predict_proba(X_test)[:, 1]
    f1 = f1_score(y_test, model.predict(X_test))
    ap = average_precision_score(y_test, y_prob)
    results.append([name, round(f1, 4), round(ap, 4)])

df_res = pd.DataFrame(results, columns=['Model', 'F1', 'AUC‑PR'])
df_res.style.background_gradient(cmap='Blues', subset=['F1', 'AUC‑PR'])

,Model,F1,AUC‑PR
0,Logistic Regression (E‑comm),0.277600,0.391100
1,Random Forest (E‑comm),0.694100,0.629300
2,XGBoost (E‑comm),0.694600,0.622900
3,Logistic Regression (Credit),0.100200,0.676800
4,Random Forest (Credit),0.822900,0.813300
5,XGBoost (Credit),0.760000,0.810500


In [11]:
import os
os.makedirs('../models', exist_ok=True)
with open('../models/fraud_xgb.pkl', 'wb') as f:
    pickle.dump(best_xgb_f, f)
with open('../models/credit_xgb.pkl', 'wb') as f:
    pickle.dump(best_xgb_c, f)
print('Models saved to ../models/')

Models saved to ../models/
